<a href="https://colab.research.google.com/github/iam-ramshakhalid/Pothole-Detection/blob/main/F26-1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Install the roboflow library (-q means "quiet", so less output)
!pip install roboflow -q

# Import the tools we need
from google.colab import userdata   # reads our hidden key from Colab Secrets
from roboflow import Roboflow       # lets us talk to Roboflow

# Read the key from Secrets (it is never written in the notebook)
api_key = userdata.get("ROBOFLOW_API_KEY")

# Connect to Roboflow using the key
rf = Roboflow(api_key=api_key)

# Choose the dataset: workspace name, then project name (from the dataset link)
project = rf.workspace("gerapothole").project("pothole-detection-yolov8")

# Choose version 1 and download it in YOLOv8 format
version = project.version(1)
dataset = version.download("yolov8")

# Show where the files were saved
print("Downloaded to:", dataset.location)

loading Roboflow workspace...
loading Roboflow project...
Downloaded to: /content/Pothole-detection-YOLOv8-1


In [4]:
import os   # tool for working with folders and files

# The folder where cell 1 saved the dataset
path = dataset.location
print("Dataset folder:", path)

# See what is inside the main folder
print("\nInside the folder:", os.listdir(path))

# Count the images and label files in each part (train, valid, test)
for split in ["train", "valid", "test"]:
    img_folder = os.path.join(path, split, "images")
    lbl_folder = os.path.join(path, split, "labels")
    if os.path.exists(img_folder):
        print(split, "-> images:", len(os.listdir(img_folder)),
              "| labels:", len(os.listdir(lbl_folder)))
    else:
        print(split, "-> folder not found")

# Print data.yaml, the file that lists the class names
print("\n--- data.yaml ---")
print(open(os.path.join(path, "data.yaml")).read())

Dataset folder: /content/Pothole-detection-YOLOv8-1

Inside the folder: ['README.dataset.txt', 'train', 'valid', 'data.yaml', 'README.roboflow.txt', 'test']
train -> images: 1245 | labels: 1245
valid -> images: 120 | labels: 120
test -> images: 62 | labels: 62

--- data.yaml ---
names:
- Potholes
nc: 1
roboflow:
  license: CC BY 4.0
  project: pothole-detection-yolov8
  url: https://universe.roboflow.com/gerapothole/pothole-detection-yolov8/dataset/1
  version: 1
  workspace: gerapothole
test: ../test/images
train: ../train/images
val: ../valid/images



In [5]:
import os

bases = {}   # will hold the "original photo name" of every image, per split

for split in ["train", "valid", "test"]:
    files = os.listdir(os.path.join(path, split, "images"))

    # Roboflow names augmented files like: 22_jpg.rf.<random code>.jpg
    # Everything before ".rf." is the original photo's name
    bases[split] = [f.split(".rf.")[0] for f in files]

    print(split, "example file names:", files[:3])
    print(split, "images:", len(files),
          "| unique original photos:", len(set(bases[split])))
    print()

# Do the same original photo appear in more than one split?
print("Shared between train and valid:", len(set(bases["train"]) & set(bases["valid"])))
print("Shared between train and test :", len(set(bases["train"]) & set(bases["test"])))
print("Shared between valid and test :", len(set(bases["valid"]) & set(bases["test"])))

# Total number of different original photos in the whole dataset
all_unique = set(bases["train"]) | set(bases["valid"]) | set(bases["test"])
print("\nTotal unique original photos:", len(all_unique))

train example file names: ['109_jpg.rf.d6f728cabfde2497202f168657d86623.jpg', '431_jpg.rf.bf09991530a447eab9cc895b40cee05c.jpg', '517_jpg.rf.4fe1fa0d7a99f94759bac7ea2102ccd7.jpg']
train images: 1245 | unique original photos: 425

valid example file names: ['205_jpg.rf.7c02f70327a1a8a5181b0f09189ba6c3.jpg', '503_jpg.rf.37350769c261f565a5586a4787d5791f.jpg', '83_jpg.rf.32f2e7335aa616c6b7fc4aa9a4c09214.jpg']
valid images: 120 | unique original photos: 120

test example file names: ['463_jpg.rf.7158f0539c28f2427c3749050e469ec7.jpg', '401_jpg.rf.e6d4c4f98170e3ffbb0bfda9841a59f4.jpg', '79_jpg.rf.3d21f60cd8dc88795545ed19eb929fca.jpg']
test images: 62 | unique original photos: 62

Shared between train and valid: 0
Shared between train and test : 0
Shared between valid and test : 0

Total unique original photos: 607


In [6]:
import os, shutil

# Make the working folders (we clean these, never the original download)
os.makedirs("work/images", exist_ok=True)
os.makedirs("work/labels", exist_ok=True)

seen = set()            # original photo names we already kept
raw_total = 0           # all images in the download
duplicate_count = 0     # extra copies we skip

for split in ["train", "valid", "test"]:
    img_folder = os.path.join(path, split, "images")
    lbl_folder = os.path.join(path, split, "labels")

    for img_name in sorted(os.listdir(img_folder)):
        raw_total += 1

        # Original photo name = everything before ".rf."
        base = img_name.split(".rf.")[0]

        # If we already kept this photo, skip this extra copy
        if base in seen:
            duplicate_count += 1
            continue
        seen.add(base)

        # Copy the image
        shutil.copy(os.path.join(img_folder, img_name), "work/images/")

        # Copy its label file (same name, but .txt)
        lbl_path = os.path.join(lbl_folder, os.path.splitext(img_name)[0] + ".txt")
        if os.path.exists(lbl_path):
            shutil.copy(lbl_path, "work/labels/")

print("Raw images in download :", raw_total)
print("Duplicates removed     :", duplicate_count)
print("Images kept in work/   :", len(os.listdir("work/images")))
print("Labels kept in work/   :", len(os.listdir("work/labels")))

Raw images in download : 1427
Duplicates removed     : 820
Images kept in work/   : 607
Labels kept in work/   : 607


In [7]:
import os    # for working with files and folders
import cv2   # OpenCV, used here to try opening each image

# Counters: we print these at the end and put them in the report
corrupt_count = 0   # broken images we delete
orphan_count = 0    # images or labels that have no partner file

# ---------- Part A: delete corrupt images ----------
# A corrupt image is one that OpenCV cannot open.
for image_file in os.listdir("work/images"):
    image_path = "work/images/" + image_file

    # Try to open the image. If it is broken, OpenCV gives back None.
    image = cv2.imread(image_path)

    if image is None:
        # Delete the broken image
        os.remove(image_path)

        # Delete its label file too. The label has the same name but ends in .txt
        # (splitext cuts off only the last ending, for example ".jpg")
        name = os.path.splitext(image_file)[0]
        label_path = "work/labels/" + name + ".txt"
        if os.path.exists(label_path):
            os.remove(label_path)

        corrupt_count = corrupt_count + 1

# ---------- Part B: delete orphans ----------
# An orphan is an image with no label file, or a label file with no image.

# B1: images that have no label file
for image_file in os.listdir("work/images"):
    name = os.path.splitext(image_file)[0]
    label_path = "work/labels/" + name + ".txt"

    if not os.path.exists(label_path):
        os.remove("work/images/" + image_file)
        orphan_count = orphan_count + 1

# B2: label files that have no image
# First, make a list of the names of all images that are left
image_names = []
for image_file in os.listdir("work/images"):
    image_names.append(os.path.splitext(image_file)[0])

# Then check every label file against that list
for label_file in os.listdir("work/labels"):
    name = os.path.splitext(label_file)[0]

    if name not in image_names:
        os.remove("work/labels/" + label_file)
        orphan_count = orphan_count + 1

# ---------- Results ----------
print("Corrupt images deleted:", corrupt_count)
print("Orphan files deleted  :", orphan_count)
print("Images left:", len(os.listdir("work/images")))
print("Labels left:", len(os.listdir("work/labels")))

Corrupt images deleted: 0
Orphan files deleted  : 0
Images left: 607
Labels left: 607
